In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

base_dir = "/home/isaac/Documents/steady/34HeCONZND_k1e-8/"

isotopes = ["he4", "c12", "n14", "o16", "ne20", "mg24", "si28", "s32", "ar36", "ca40"]
# read isotope data into dict
iso_data = {
    iso: pd.read_csv(f"{base_dir}znd_xa_{iso}_x.xq", sep = r"\s+", skiprows = 2, usecols = [0,1], engine = "c")
    for iso in isotopes
}
# read thermodynamic data in into dict
thermo_vars = ["temp", "rho", "pg", "u", "mach", "bind"]
thermo_data = {
    thermo: pd.read_csv(f"{base_dir}znd_{thermo}_x.xq", delimiter = ' ', skiprows = 2, usecols = [0,1], engine = "c")
    for thermo in thermo_vars
}


In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
isotope_labels = {}
for iso, df in iso_data.items():
    x, xa = df.iloc[:,0], df.iloc[:,1]
    ax.plot(x, xa, lw = 1, label = isotope_labels.get(iso, iso.capitalize()))

ax.plot(thermo_data['mach']['x'], thermo_data['mach']['mach'], color="black", label='M', lw=1)

ax2 = ax.twinx()
ax2.plot(thermo_data['temp']['x'], thermo_data['temp']['temp'], color = "red", ls = "-.", lw = 1, label='T')

ax.set_xscale("log")
ax.set_xlim(1e-1, 1e7)
ax.set_ylim(0, 1)
ax2.set_ylim(1.71e9, 1.925e9)

ax.grid(True, which='major', axis='x', color='gray', ls='-', alpha=0.6)
ax.grid(True, which='minor', axis='x', color='gray', ls=':', alpha=0.6)
ax.tick_params(axis='x', which='major', direction='in', top=True, length=8)
ax.tick_params(axis='x', which='minor', direction='in', top=True, length=4)
ax.tick_params(axis='y', which='major', direction='in', length=6)
ax2.tick_params(axis='y', which='major', direction='in', length=6)

ax.text(0.2, 0.92, rf"D = {thermo_data['u']['u'].iloc[0]:e} cm $\cdot s^{{-1}}$     $l_{{ind}}$ $\simeq 13.7$ cm", fontsize=12)

ax.set_title(r'34 Isotope  $\kappa = 10^{-8}$')
ax.set_xlabel('x (cm)')
ax.set_ylabel(r'$X_i$, M')
ax2.set_ylabel('T (K)')

fig.legend(loc='upper center', bbox_to_anchor=(0.5, -0.001), ncol=6, frameon=False)
fig.tight_layout()

#plt.savefig("figures/34isok8_steady", dpi=300, transparent=True, bbox_inches='tight')

In [ ]:
# analysis stuff

print(f'''
Initial
T = {thermo_data['temp']['temp'].iloc[0]:e} K
rho = {thermo_data['rho']['rho'].iloc[0]:e} g/cm^3
P = {thermo_data['pg']['pg'].iloc[0]:e} Ba
Post Shock State
T = {thermo_data['temp']['temp'].iloc[1]:e} K
rho = {thermo_data['rho']['rho'].iloc[1]:e} g/cm^3
P = {thermo_data['pg']['pg'].iloc[1]:e} Ba
u_mat = {thermo_data['u']['u'].iloc[1]:e} cm/s
D = {thermo_data['u']['u'].iloc[0]:e} K
''')


# central difference method
T = list(thermo_data['temp']['temp'][3500:4100])
x = list(thermo_data['temp']['x'][3500:4100])
n = len(T)
dT_dx = [0.0] * n

for i in range(1, n-1):
    dT_dx[i] = (T[i+1] - T[i-1]) / (x[i+1]-x[i-1]) #central difference

dT_dx[0] = (T[2] - T[0]) / (x[2] - x[0])

max_dT = dT_dx[0]
max_idx = 0

for i in range(1, n):
    if dT_dx[i] > max_dT:
        max_dT = dT_dx[i]
        max_idx = i

print(f"Max T_dot: {max_dT:.4f} T/cm")
print(f"x_max_T_dot: {x[max_idx]:.4f} cm")


# finite difference method
x = np.array(thermo_data['temp']['x'][3400:4200])
temp = np.array(thermo_data['temp']['temp'][3400:4200])

# calculate the rate of change
slopes = np.gradient(temp, x)

# index of the max slope
max_slope_idx = np.argmax(slopes)

max_slope = slopes[max_slope_idx]
max_slope_x = x[max_slope_idx]

print(f"Maximum Slope (dT/dx): {max_slope:.4f}")
print(f"Occurs at x: {max_slope_x:.4f}")